# ============================================================
# **EMPLOYEE ATTRITION PREDICTION - LOGISTIC REGRESSION MODEL**
# ============================================================



## **1. IMPORT LIBRARIES**


In [ ]:
import pandas as pd
import numpy as np

# Database
from sqlalchemy import create_engine

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Saving Model
import joblib

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(style="whitegrid")




## **2. LOAD DATA FROM MYSQL**


In [ ]:
username = "root"
password = "pricass00"
host = "localhost"
database = "hr_analytics"

try:
    engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}/{database}")
    df = pd.read_sql("SELECT * FROM employee_attrition", con=engine)
    print(f"Data Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")
except Exception as e:
    print("Error loading data:", e)
    exit()




## **3. DATA CLEANING & FEATURE ENGINEERING**


In [ ]:

# 3.1 Create Target Variable
df["attrition_flag"] = df["attrition"].map({"Yes": 1, "No": 0})

# 3.2 Drop Columns Not Useful for Modeling
cols_to_drop = [
    "attrition",
    "employeecount",
    "over18",
    "standardhours",
    "employeenumber"
]
df_ml = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 3.3 One-Hot Encoding
df_ml = pd.get_dummies(df_ml, drop_first=True)

print(f"Dataset Ready for Modeling: {df_ml.shape}")




## **4. TRAIN-TEST SPLIT**


In [ ]:
X = df_ml.drop("attrition_flag", axis=1)
y = df_ml["attrition_flag"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)




## **5. FEATURE SCALING**


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)




## **6. MODEL TRAINING**


In [ ]:
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)




## **7. MODEL EVALUATION**


In [ ]:
accuracy = round(accuracy_score(y_test, y_pred) * 100, 2)
print(f"\nModel Accuracy: {accuracy}%")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))




## **8. FEATURE IMPORTANCE**


In [ ]:
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values(by="coefficient", ascending=False)

print("\nTop 10 Factors Increasing Attrition:")
print(coef_df.head(10))

print("\nTop 10 Factors Reducing Attrition:")
print(coef_df.tail(10))




## **9. SAVE MODEL FOR DEPLOYMENT**


In [ ]:
joblib.dump(model, "logistic_model_attrition.pkl")
joblib.dump(scaler, "scaler_attrition.pkl")

print("\nModel and Scaler saved successfully.")
